# 02 · Delivery lateness

**Question:** How often are delivered orders late, by how much, and does the
late rate vary over time and by customer state?

"Late" = delivered on a later date than the estimated date (see
`sql/01_order_delivery.sql`). Only delivered orders are included.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

# Notebooks run from notebooks/, so add src/ to the import path.
sys.path.insert(0, str(Path.cwd().parent / "src"))
from db import query

sns.set_theme(style="whitegrid")

## 1. Late rate

**Question:** What share of delivered orders arrived after the estimated date?

In [ ]:
query("""
    SELECT is_late,
           COUNT(*)                                        AS orders,
           ROUND(COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (), 4) AS share
    FROM order_delivery
    WHERE is_delivered
    GROUP BY is_late
""")

**Your notes:**

- 

## 2. How early or late

**Question:** How are `days_late` distributed? Negative = early, positive = late.
The plot is clipped to ±60 days so the bulk is readable; the table shows the full range.

In [ ]:
days = query("SELECT days_late FROM order_delivery WHERE is_delivered")

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(days["days_late"].clip(-60, 60), binwidth=1, ax=ax)
ax.axvline(0, color="black", linewidth=1)
ax.set(title="Days delivered after the estimate (clipped to ±60)", xlabel="days_late", ylabel="Orders")
plt.show()

days["days_late"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])

In [ ]:
# Among late orders only: how late?
days.loc[days["days_late"] > 0, "days_late"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

**Your notes:**

- 

## 3. Late rate over time

**Question:** Does the late rate change by purchase month? Months with few
orders are noisy; check the `orders` column before reading the line.

In [ ]:
by_month = query("""
    SELECT date_trunc('month', order_purchase_timestamp)::date AS month,
           COUNT(*)                                            AS orders,
           AVG(is_late::int)                                   AS late_rate
    FROM order_delivery
    WHERE is_delivered
    GROUP BY 1
    ORDER BY 1
""")

ax = by_month.plot(x="month", y="late_rate", marker="o", legend=False, figsize=(10, 4))
ax.set(title="Late rate by purchase month", xlabel="", ylabel="Late rate")
plt.show()
by_month

**Your notes:**

- 

## 4. Late rate by customer state

**Question:** Does the late rate differ by the customer's state? States with
few orders are noisy, so `orders` is shown next to the rate.

In [ ]:
by_state = query("""
    SELECT customer_state,
           COUNT(*)            AS orders,
           AVG(is_late::int)   AS late_rate
    FROM order_delivery
    WHERE is_delivered
    GROUP BY customer_state
    ORDER BY late_rate DESC
""")

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=by_state, y="customer_state", x="late_rate", color="steelblue", ax=ax)
ax.set(title="Late rate by customer state", xlabel="Late rate", ylabel="")
plt.show()
by_state

**Your notes:**

- 

## Decisions from this notebook

-